# Phase 1 · 01 — Tool calling **from scratch**

Before touching any library, let's see what "tool calling" really is.

**An LLM is just text in → text out. It cannot run your code.** So how does an
"AI agent" run functions? By a convention *we* invent:

1. We **teach the model a format** for *asking* to run a function
   (e.g. a line like `TOOL_CALL: {"name": "multiply", "args": {...}}`).
2. The model outputs text. If it contains that format, it's *requesting* a call.
3. **Our code (the "harness")** parses the request, runs the real Python
   function, and pastes the result back into the conversation.
4. We call the model again so it can use the result. Repeat until it's done.

That loop — prompt, parse, dispatch, feed back — is the *entire* mechanism.
Every agent framework and every provider's "tool calling" is a polished version
of it. We'll build it by hand, first driven by a **scripted fake model** (so the
whole thing runs with zero dependencies and zero API key), then swap in a real
LLM over raw HTTP.

## The four ingredients

1. **A tool registry** — the real functions, plus a short description of each.
2. **A protocol prompt** — text that teaches the model how to request a tool.
3. **A parser** — detects a tool request in the model's raw text.
4. **The agent loop** — ties it together.

### 1. The tool registry

In [ ]:
def add(a, b):
    """Add two numbers."""
    return a + b

def multiply(a, b):
    """Multiply two numbers."""
    return a * b

# name -> {the function, a description, the arguments} so we can both
# (a) tell the model about it, and (b) actually call it by name.
TOOLS = {
    "add":      {"fn": add,      "description": "Add two numbers.",      "args": "a (number), b (number)"},
    "multiply": {"fn": multiply, "description": "Multiply two numbers.", "args": "a (number), b (number)"},
}

print(add(2, 3), multiply(4, 5))

### 2. The protocol prompt

The model can't see our Python, so we *describe* the tools in plain text and
teach it the exact format to request one. The format is our choice — it just has
to be something we can reliably parse. Here: a line starting with `TOOL_CALL:`
followed by JSON.

In [ ]:
def build_system_prompt(tools):
    lines = ["You are an assistant that can call tools to do arithmetic.", "", "Available tools:"]
    for name, t in tools.items():
        lines.append(f'  - {name}({t["args"]}): {t["description"]}')
    lines += [
        "",
        "When you need a tool, reply with EXACTLY one line and nothing else:",
        '  TOOL_CALL: {"name": "<tool>", "args": {"a": <number>, "b": <number>}}',
        "",
        "You will then receive a line starting with TOOL_RESULT.",
        "When you have the final answer, reply in plain English with NO TOOL_CALL line.",
    ]
    return "\n".join(lines)

print(build_system_prompt(TOOLS))

### 3. The parser

Look at the model's raw text and pull out a tool request if there is one.
Returns `(name, args)` or `None`.

In [ ]:
import json, re

def parse_tool_call(text):
    m = re.search(r'TOOL_CALL:\s*(\{.*\})', text, re.DOTALL)
    if not m:
        return None                     # no tool requested -> this is the final answer
    obj = json.loads(m.group(1))
    return obj["name"], obj["args"]

# quick check
print(parse_tool_call('TOOL_CALL: {"name": "add", "args": {"a": 1, "b": 2}}'))
print(parse_tool_call('The answer is 3.'))

### A scripted "fake model"

To see the loop work with **no API key**, we stand in a dumb, scripted model.
It has no real intelligence — its only job is to exercise our harness by emitting
`TOOL_CALL` lines in the right format. (It *does* read the previous
`TOOL_RESULT`, so step 2 genuinely uses step 1's output.) The real brain comes
later; the harness around it never changes.

In [ ]:
def fake_model(messages):
    # How many tool results have come back so far?
    results = [m for m in messages if m["content"].startswith("TOOL_RESULT")]
    n = len(results)
    if n == 0:
        # First: request the multiplication.
        return 'I will multiply first.\nTOOL_CALL: {"name": "multiply", "args": {"a": 12.5, "b": 8}}'
    if n == 1:
        # Read the real result of step 1 and use it in step 2.
        prev = float(results[-1]["content"].split(":")[-1])
        return f'Now add 100.\nTOOL_CALL: {{"name": "add", "args": {{"a": {prev}, "b": 100}}}}'
    # Done: no TOOL_CALL line -> final answer.
    prev = float(results[-1]["content"].split(":")[-1])
    return f"The final answer is {prev}."

### 4. The agent loop

Call the model. If it asked for a tool, run it, append the result, and loop so
the model can see it. If it didn't, we're done. The `while` loop is what lets a
single question trigger *several* tool calls in sequence.

In [ ]:
def run_agent(user_message, model_fn, max_steps=8):
    messages = [
        {"role": "system", "content": build_system_prompt(TOOLS)},
        {"role": "user",   "content": user_message},
    ]
    for _ in range(max_steps):
        text = model_fn(messages)                       # 1. ask the model
        messages.append({"role": "assistant", "content": text})

        call = parse_tool_call(text)                    # 2. did it request a tool?
        if call is None:
            return text                                 #    no -> final answer

        name, args = call                               # 3. run the real function
        result = TOOLS[name]["fn"](**args)
        print(f"  [harness] ran {name}({args}) -> {result}")

        # 4. feed the result back as the next turn, then loop
        messages.append({"role": "user", "content": f"TOOL_RESULT for {name}: {result}"})
    return "(stopped: too many steps)"

In [ ]:
answer = run_agent("What is 12.5 times 8, and then add 100?", fake_model)
print("\nFinal answer:", answer)

## Now swap in a **real** LLM — same loop

Nothing about the harness changes. We only replace `fake_model` with a function
that asks a real model. To keep this first-principles, we call the model over
**raw HTTP** (`requests`) with a plain chat request — **no** provider "tools"
feature. The model emits `TOOL_CALL` lines purely because our system prompt
taught it to, and our own parser handles them.

> Needs `OPENAI_API_KEY` in the environment. `export OPENAI_API_KEY="sk-..."`
> before launching Jupyter, then restart the kernel.

In [ ]:
import os, requests

MODEL = "gpt-4o-mini"   # any small chat model works; we do the tool logic ourselves

def real_model(messages):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
            "Content-Type": "application/json",
        },
        json={"model": MODEL, "messages": messages, "temperature": 0},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  Set OPENAI_API_KEY and restart the kernel to run the next cell.")

In [ ]:
answer = run_agent("What is 12.5 times 8, and then add 100?", real_model)
print("\nFinal answer:", answer)

## Recap

You built the whole mechanism by hand:

- a **registry**, a **protocol prompt**, a **parser**, and a **loop**;
- swapping the fake model for a real one changed *nothing* about the harness.

This is exactly what agent frameworks do under the hood. The catch: we parse
free-form text, which is brittle — a slightly-off `TOOL_CALL` line breaks.

**Next → `02_tool_calling_with_sdk.ipynb`:** how SOTA fixes that — the provider
fine-tunes the model to emit *structured* tool calls and returns them as typed
fields, so there's nothing to hand-parse.